In [1]:
pip install datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
import datasets,huggingface_hub,os

In [3]:
huggingface_hub.login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from datasets import load_dataset

In [5]:
dataset = load_dataset("bookcorpus", split="train")
print(len(dataset))

README.md: 0.00B [00:00, ?B/s]

bookcorpus.py: 0.00B [00:00, ?B/s]

The repository for bookcorpus contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bookcorpus.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/74004228 [00:00<?, ? examples/s]

74004228


In [6]:
subset = dataset.select(range(0, len(dataset), 7))
print(len(subset))

10572033


In [7]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase
from tokenizers.processors import TemplateProcessing

def train_bpe_tokenizer(vocab_size):

    # Initialize tokenizer
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

    # Normalizer
    tokenizer.normalizer = Lowercase()

    # PreTokenizer
    tokenizer.pre_tokenizer = Whitespace()

    # Trainer
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[GO]", "[UNK]", "[PAD]", "[EOS]"]
    )

    # Iterator over dataset
    def batch_iterator():
        for i in range(0, len(subset), 1000):
            yield subset[i:i+1000]["text"]

    tokenizer.train_from_iterator(batch_iterator(), trainer)

    return tokenizer

In [8]:
text = "SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24."

In [24]:
tokenizer_5k = train_bpe_tokenizer(32000)
output_5k = tokenizer_5k.encode(text)

print(output_5k.tokens)
print(output_5k.ids)

['se', 'bi', 'study', 'finds', '9', '3', '%', 'of', 'individual', 'f', '&', 'o', 'traders', 'made', 'losses', 'between', 'fy', '22', 'and', 'fy', '24', '.']
[108, 1386, 3463, 6389, 33, 27, 13, 100, 5529, 52, 14, 61, 20728, 440, 16634, 716, 4560, 8240, 91, 4560, 8407, 22]


In [25]:
print(len(output_5k.tokens))
print(len(output_5k.ids))

22
22


In [26]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file("hopper.json")

In [27]:
encoding = tokenizer.encode(text)

In [28]:
print(encoding.tokens)
print("Token count:", len(encoding.tokens))

['seb', '##i', 'study', 'finds', '9', '##3', '%', 'of', 'individual', 'f', '&', 'o', 'traders', 'made', 'losses', 'between', 'f', '##y', '##22', 'and', 'f', '##y', '##2', '##4', '.']
Token count: 25


In [17]:
tokenizer.add_tokens(["FY"])

1

In [20]:
!pip install transformers

In [29]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

print("BERT special tokens:", bert_tok.special_tokens_map)
print("GPT2 special tokens:", gpt2_tok.special_tokens_map)


BERT special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
GPT2 special tokens: {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}


In [30]:
imdb = load_dataset("imdb")

# drop unsupervised
train = imdb["train"]
test = imdb["test"]

In [31]:
hopper = Tokenizer.from_file("hopper.json")
bert = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2 = AutoTokenizer.from_pretrained("gpt2")

def count_tokens_hf(tokenizer):
    total = 0
    for sample in train:
        total += len(tokenizer(sample["text"]).input_ids)
    for sample in test:
        total += len(tokenizer(sample["text"]).input_ids)
    return total

def count_tokens_tokenizers(tokenizer):
    total = 0
    for sample in train:
        total += len(tokenizer.encode(sample["text"]).ids)
    for sample in test:
        total += len(tokenizer.encode(sample["text"]).ids)
    return total

print("Custom 32K:", count_tokens_tokenizers(tokenizer_5k))
print("Hopper:", count_tokens_tokenizers(hopper))
print("BERT:", count_tokens_hf(bert))
print("GPT2:", count_tokens_hf(gpt2))

Custom 32K: 15235160


Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors


Hopper: 13526933


Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


BERT: 15516058
GPT2: 14812432
